# SINDy-based Dengue Compartmental Model Identification

**Reference:** Puspita et al. (2023), *Physica A 625*, 129019

Identifies compartmental ODE structure from weekly hospitalized dengue case data only.
Six variants covering delayed SIR/SEIR, semi-analytic reconstruction (Puspita method),
cumulative auxiliary state, and Takens delay embedding.


## 0. Imports & Configuration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.ndimage import gaussian_filter1d
from scipy.interpolate import UnivariateSpline
from scipy.integrate import cumulative_trapezoid
from scipy.optimize import minimize
from itertools import combinations_with_replacement, product as iproduct
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
print("Imports OK")


## 1. Configuration
Set your data path and parameters here.

In [ ]:
# ── USER SETTINGS ─────────────────────────────────────────────────────────
DATA_PATH   = None          # set to 'your_file.csv' when you have real data
                            # CSV: single column of weekly hospitalized cases
CSV_HEADER  = True          # True if CSV has a header row

N_WEEKS     = 312           # used only for synthetic placeholder

# Population (flexible — set per year or approximate)
Nh = 500_000                # adjust to your region's population

# Epidemiological priors (Puspita 2023 / dengue literature)
DELTA = 7 / 4               # incubation rate (week^-1)
GAMMA = 1.0                 # recovery rate (week^-1)
MU_H  = 1 / (70 * 52)      # natural death rate (week^-1)
MU_V  = 0.5                 # mosquito death rate (week^-1)
OMEGA = 2.5                 # mosquito-human transmission ratio

# SINDy settings
POLY_ORDER  = 2             # library polynomial order (1 or 2)
THRESHOLD   = 0.05          # STLS sparsity threshold (normalized scale)
SIGMA       = 1.5           # Gaussian smoothing sigma (weeks)

# Multi-logistic fit (V3/V4)
N_LOGISTIC  = 8             # number of logistic terms (Puspita uses 8)
N_RESTARTS  = 30            # random restarts for nonlinear fitting

# Grid search ranges
TAU_GRID    = [1, 2, 3, 4]  # delay grid for V1 (weeks)
TAU1_GRID   = [1, 2, 3]     # incubation delay grid for V2
TAU2_GRID   = [1, 2, 3]     # recovery delay grid for V2
EMB_TAU     = [1, 2, 3]     # embedding delay grid for V6
EMB_DIM     = [3, 4, 5]     # embedding dimension grid for V6

print("Configuration set.")


## 2. Data Loading & Synthetic Placeholder

In [ ]:
def generate_synthetic(n_weeks=312, seed=42, noise_level=0.15):
    """Simulate weekly dengue cases using Puspita SEIR-SI reduced model."""
    rng = np.random.default_rng(seed)
    _Nh  = 362_202
    _Ah  = 7_233 / 52
    _mu  = _Ah / _Nh
    _d   = 7 / 4
    _g   = 1.0
    _muv = 0.5
    _om  = 2.5
    _Nv  = _Nh
    t    = np.arange(n_weeks, dtype=float)
    bh   = 0.45 + 0.08*np.sin(2*np.pi*t/52) + 0.05*np.sin(4*np.pi*t/52 + 0.5)

    def rhs(s, b):
        Sh, Eh, Ih, Rh = s
        force = b*_om*_Nv*Ih*Sh / ((b*_om*Ih + _muv*_Nh)*_muv*_Nh)
        return np.array([
            _Ah - force - _mu*Sh,
            force - (_d+_mu)*Eh,
            _d*Eh - (_g+_mu)*Ih,
            _g*Ih - _mu*Rh
        ])

    state = np.array([_Nh-13, 5.0, 13.0, 0.0])
    I_true = np.zeros(n_weeks)
    for i in range(n_weeks):
        k1=rhs(state,bh[i]); k2=rhs(state+.5*k1,bh[i])
        k3=rhs(state+.5*k2,bh[i]); k4=rhs(state+k3,bh[i])
        state = np.maximum(state+(k1+2*k2+2*k3+k4)/6, 0)
        I_true[i] = state[2]
    I_obs = rng.poisson(np.maximum(I_true*(1+noise_level*rng.standard_normal(n_weeks)), 0.5))
    return t, I_obs.astype(float)


def load_data():
    if DATA_PATH is not None:
        import pandas as pd
        df  = pd.read_csv(DATA_PATH, header=0 if CSV_HEADER else None)
        I   = df.iloc[:, 0].values.astype(float)
        t   = np.arange(len(I), dtype=float)
        print(f"Loaded real data: {len(I)} weeks")
    else:
        print("No DATA_PATH set — using synthetic placeholder.")
        t, I = generate_synthetic(N_WEEKS)
    return t, I


t_raw, I_raw = load_data()
print(f"  T={len(t_raw)} weeks, mean={I_raw.mean():.1f}, max={I_raw.max():.0f}")


## 3. Smoothing & Cumulative K(t)

In [ ]:
def smooth_gaussian(I, sigma=1.5):
    return gaussian_filter1d(I.astype(float), sigma=sigma)

def smooth_spline(t, I, sf=None):
    sf  = sf or len(I)*np.var(I)*0.1
    spl = UnivariateSpline(t, I, s=sf, k=4)
    return spl(t), spl

def compute_K(t, I_smooth):
    """K(t) = cumulative integral of I, K(0)=I(0) (Puspita Eq. 9)."""
    K = cumulative_trapezoid(I_smooth, t, initial=0)
    K += I_smooth[0]
    return K

# Apply smoothing
I_gauss = smooth_gaussian(I_raw, sigma=SIGMA)
I_spl, spline = smooth_spline(t_raw, I_raw)
I_smooth = I_spl          # default: spline

K = compute_K(t_raw, I_smooth)

# Plot
fig, axes = plt.subplots(3, 1, figsize=(13, 8), sharex=True)
axes[0].bar(t_raw, I_raw, color='steelblue', alpha=0.4, label='Raw')
axes[0].plot(t_raw, I_gauss, 'g-', lw=1.5, label='Gaussian')
axes[0].plot(t_raw, I_spl,  'r-', lw=2,   label='Spline')
axes[0].set_ylabel('Cases'); axes[0].legend(fontsize=8)
axes[0].set_title('Weekly Hospitalized Cases I(t)')
axes[1].plot(t_raw, K, 'darkgreen', lw=2)
axes[1].set_ylabel('K(t)'); axes[1].set_title('Cumulative K(t) = I + R')
dI = gaussian_filter1d(np.gradient(I_smooth, t_raw), sigma=SIGMA)
axes[2].plot(t_raw, dI, 'purple', lw=1.5)
axes[2].axhline(0, color='k', lw=0.5)
axes[2].set_ylabel('dI/dt'); axes[2].set_xlabel('Week')
plt.tight_layout(); plt.show()


## 4. SINDy Core: Library & STLS

In [ ]:
def build_library(X, poly_order=2, include_bias=True,
                  custom_fns=None, custom_names=None):
    """Build SINDy feature library Theta(X)."""
    n, m = X.shape
    cols, names = [], []
    if include_bias:
        cols.append(np.ones(n)); names.append('1')
    for i in range(m):
        cols.append(X[:, i])
    if poly_order >= 2:
        for i, j in combinations_with_replacement(range(m), 2):
            cols.append(X[:, i]*X[:, j])
    if custom_fns:
        for fn, nm in zip(custom_fns, custom_names):
            cols.append(fn(X)); names.append(nm)
    Theta = np.column_stack(cols)
    return Theta


def make_feature_names(state_names, poly_order=2, include_bias=True):
    m = len(state_names)
    names = []
    if include_bias: names.append('1')
    names += list(state_names)
    if poly_order >= 2:
        for i, j in combinations_with_replacement(range(m), 2):
            names.append(f'{state_names[i]}·{state_names[j]}')
    return names


def estimate_derivatives(t, X, sigma=1.5):
    dX = np.gradient(X, t, axis=0)
    for j in range(X.shape[1]):
        dX[:, j] = gaussian_filter1d(dX[:, j], sigma=sigma)
    return dX


def stls(Theta, dX, threshold=0.05, max_iter=20):
    """Sequentially Thresholded Least Squares with column normalization."""
    Theta = np.nan_to_num(Theta); dX = np.nan_to_num(dX)
    n, p  = Theta.shape
    q     = dX.shape[1]
    norms = np.linalg.norm(Theta, axis=0)
    norms[norms < 1e-12] = 1.0
    Tn    = Theta / norms
    Xi_n  = np.linalg.lstsq(Tn, dX, rcond=None)[0]
    for _ in range(max_iter):
        small = np.abs(Xi_n) < threshold
        Xi_n[small] = 0.0
        for j in range(q):
            big = ~small[:, j]
            if big.sum() == 0: continue
            Xi_n[big, j] = np.linalg.lstsq(Tn[:, big], dX[:, j], rcond=None)[0]
    return Xi_n / norms[:, None]


def aic_bic(Theta, dX, Xi):
    n    = Theta.shape[0]
    dXh  = Theta @ Xi
    rss  = np.sum((dX - dXh)**2)
    s2   = max(rss/(n*dX.shape[1]), 1e-12)
    ll   = -0.5*n*dX.shape[1]*(np.log(2*np.pi*s2)+1)
    k    = np.sum(Xi != 0)
    return 2*k - 2*ll,  k*np.log(n) - 2*ll


def run_sindy(t, X, state_names, threshold=THRESHOLD,
              poly_order=POLY_ORDER, sigma=SIGMA, verbose=True):
    Theta  = build_library(X, poly_order=poly_order)
    fnames = make_feature_names(state_names, poly_order=poly_order)
    dX     = estimate_derivatives(t, X, sigma=sigma)
    Xi     = stls(Theta, dX, threshold=threshold)
    aic, bic = aic_bic(Theta, dX, Xi)
    if verbose:
        print("Identified equations:")
        for j, sn in enumerate(state_names):
            terms = [f"({Xi[i,j]:.4f})·{fnames[i]}"
                     for i in range(len(fnames)) if abs(Xi[i,j]) > 1e-8]
            print(f"  d{sn}/dt = {' + '.join(terms) or '0'}")
        print(f"  AIC={aic:.1f}  BIC={bic:.1f}  nonzero={np.sum(Xi!=0)}")
    return Xi, fnames, Theta, dX, aic, bic

print("SINDy core defined.")


## 5. Compartment Builders (V1–V6)

In [ ]:
def clip_pos(x, label=""):
    n = np.sum(x < 0)
    if n: print(f"  [clip] {label}: {n} negative → 0")
    return np.maximum(x, 0)


# ── V1: Delayed SIR ────────────────────────────────────────────────────────
def build_V1(I, t=t_raw, Nh=Nh, tau=1):
    """S̃ absorbs E; R̃(t) = I(t-tau)."""
    n = len(I)
    R = np.zeros(n); R[tau:] = I[:-tau] if tau > 0 else I
    S = clip_pos(Nh - I - R, 'V1 S̃')
    X = np.column_stack([S, I, R])[tau:]
    return t[tau:], X, ['S̃', 'I', 'R̃']


# ── V2: Delayed SEIR ───────────────────────────────────────────────────────
def build_V2(I, t=t_raw, Nh=Nh, tau1=1, tau2=1, gamma=GAMMA, mu_h=MU_H):
    """E(t)≈I(t-tau1); R from numerical integral; S residual."""
    n  = len(I); dt = t[1]-t[0]
    E  = np.zeros(n); E[tau1:] = I[:-tau1] if tau1 > 0 else I
    R  = np.zeros(n)
    for k in range(1, n):
        R[k] = R[k-1] + dt*(gamma*I[k-1] - mu_h*R[k-1])
    R = clip_pos(R, 'V2 R')
    S = clip_pos(Nh - E - I - R, 'V2 S')
    tm = max(tau1, tau2)
    return t[tm:], np.column_stack([S,E,I,R])[tm:], ['S','E','I','R']


# ── Multi-logistic fit for V3/V4 ───────────────────────────────────────────
def multi_logistic(t, p, ell=N_LOGISTIC):
    K = np.full_like(t, p[0], dtype=float)
    for i in range(ell):
        a,b,c = p[1+3*i], p[2+3*i], p[3+3*i]
        z = np.clip(-b*t + c, -500, 500)
        K += a / (1 + np.exp(z))
    return K

def d_multi_logistic(t, p, ell=N_LOGISTIC):
    dK = np.zeros_like(t, dtype=float)
    for i in range(ell):
        a,b,c = p[1+3*i], p[2+3*i], p[3+3*i]
        z  = np.clip(-b*t + c, -500, 500)
        ez = np.exp(z)
        dK += a*b*ez / (1+ez)**2
    return dK

def fit_multi_logistic(t, K_data, ell=N_LOGISTIC, restarts=N_RESTARTS, seed=0):
    """Fit K(t) to multi-logistic sum (Puspita Eq. 16)."""
    rng = np.random.default_rng(seed)
    n_p = 1 + 3*ell
    def loss(p): return np.mean((multi_logistic(t, p, ell) - K_data)**2)
    best = None
    for _ in range(restarts):
        p0 = rng.normal(0, 1, n_p); p0[0] = K_data[0]
        try:
            r = minimize(loss, p0, method='Nelder-Mead',
                         options={'maxiter': 50_000, 'xatol':1e-6,'fatol':1e-6})
            if best is None or r.fun < best.fun: best = r
        except: pass
    p = best.x
    Kf  = lambda tt: multi_logistic(tt, p, ell)
    dKf = lambda tt: d_multi_logistic(tt, p, ell)
    return p, Kf, dKf, np.sqrt(best.fun)


# ── V3: Semi-analytic SEIR (Puspita Eqs 11-14) ────────────────────────────
def build_V3(I, t=t_raw, Nh=Nh, K_func=None, dK_func=None,
             delta=DELTA, gamma=GAMMA, mu_h=MU_H):
    """All compartments from analytic formulas driven by K(t)."""
    Kv  = K_func(t); dKv = dK_func(t)
    E   = clip_pos((dKv + mu_h*Kv)/delta, 'V3 E')          # Eq.11
    ig  = np.exp((gamma+mu_h)*t)*(mu_h*Kv + dKv)
    Ih  = clip_pos(np.exp(-(gamma+mu_h)*t)*(I[0] +
          cumulative_trapezoid(ig, t, initial=0)), 'V3 I')  # Eq.12
    igR = Ih*np.exp(mu_h*t)
    R   = clip_pos(np.exp(-mu_h*t)*gamma*
          cumulative_trapezoid(igR, t, initial=0), 'V3 R')  # Eq.13
    S   = clip_pos(Nh - E - Ih - R, 'V3 S')                # Eq.14
    return t, np.column_stack([S,E,Ih,R]), ['S','E','I','R']


# ── V4: Semi-analytic + cumulative C ───────────────────────────────────────
def build_V4(I, t=t_raw, Nh=Nh, K_func=None, dK_func=None,
             delta=DELTA, gamma=GAMMA, mu_h=MU_H):
    """V3 + C(t)=∫I dt as auxiliary state (dC/dt = I is library constraint)."""
    _, X3, sn3 = build_V3(I, t, Nh, K_func, dK_func, delta, gamma, mu_h)
    C = cumulative_trapezoid(I, t, initial=0)[:len(X3)]
    return t[:len(X3)], np.column_stack([X3, C]), sn3+['C']


# ── V5: Cumulative-only ────────────────────────────────────────────────────
def build_V5(I, t=t_raw, sigma=SIGMA):
    """States: [I, C, dC] — no compartment assumption."""
    C  = cumulative_trapezoid(I, t, initial=0)
    dC = gaussian_filter1d(np.gradient(C, t), sigma=sigma)
    return t, np.column_stack([I, C, dC]), ['I', 'C', 'dC']


# ── V6: Delay embedding (Takens) ───────────────────────────────────────────
def build_V6(I, t=t_raw, tau=1, embedding_dim=4):
    """Reconstruct attractor from I(t) using delay coordinates."""
    n, tm = len(I), (embedding_dim-1)*tau
    mat = np.column_stack([I[tm-d*tau: n-d*tau if d>0 else n]
                           for d in range(embedding_dim)])
    names = ['I(t)'] + [f'I(t-{d*tau}w)' for d in range(1, embedding_dim)]
    return t[tm:], mat, names

print("Compartment builders defined.")


## 6. Grid Search (Outer Loop)

In [ ]:
def grid_search(build_fn, param_grid, verbose=True):
    """
    Outer loop over hyperparameter grid; SINDy in inner loop.
    build_fn(**params) -> (t, X, state_names)
    Returns best_result dict and sorted all_results list.
    """
    keys   = list(param_grid.keys())
    combos = list(iproduct(*param_grid.values()))
    results = []
    for combo in combos:
        params = dict(zip(keys, combo))
        try:
            t_, X_, sn_ = build_fn(**params)
            Xi_, fn_, Th_, dX_, aic_, bic_ = run_sindy(
                t_, X_, sn_, verbose=False)
            results.append(dict(params=params, Xi=Xi_, feat_names=fn_,
                                aic=aic_, bic=bic_, t=t_, X=X_,
                                state_names=sn_, Theta=Th_, dX=dX_))
            if verbose:
                ps = ', '.join(f'{k}={v}' for k,v in params.items())
                print(f"  [{ps}]  AIC={aic_:.1f}  nz={np.sum(Xi_!=0)}")
        except Exception as e:
            if verbose: print(f"  {params} FAILED: {e}")
    results.sort(key=lambda r: r['aic'])
    if verbose and results:
        print(f"  → Best: {results[0]['params']}  AIC={results[0]['aic']:.1f}")
    return results[0], results

print("Grid search defined.")


## 7. Diagnostic Plots

In [ ]:
def plot_compartments(t, X, state_names, title=""):
    colors = ['navy','darkorange','crimson','forestgreen','purple','brown']
    fig, axes = plt.subplots(X.shape[1], 1,
                             figsize=(13, 2.4*X.shape[1]), sharex=True)
    if X.shape[1]==1: axes=[axes]
    fig.suptitle(title, fontsize=12)
    for j,(ax,nm) in enumerate(zip(axes, state_names)):
        ax.plot(t, X[:,j], color=colors[j%len(colors)], lw=2)
        ax.set_ylabel(nm); ax.grid(alpha=0.3)
    axes[-1].set_xlabel('Week')
    plt.tight_layout(); plt.show()


def plot_sindy_fit(t, X, Theta, Xi, dX, state_names, title=""):
    dXh = Theta @ Xi
    n   = X.shape[1]
    fig, axes = plt.subplots(n, 1, figsize=(13, 2.4*n), sharex=True)
    if n==1: axes=[axes]
    fig.suptitle(title, fontsize=12)
    for j,(ax,nm) in enumerate(zip(axes, state_names)):
        ax.plot(t, dX[:,j], 'k-',  lw=1.5, alpha=0.7, label='estimated')
        ax.plot(t, dXh[:,j],'r--', lw=1.5, label='SINDy')
        r2 = 1 - np.var(dX[:,j]-dXh[:,j])/(np.var(dX[:,j])+1e-12)
        ax.set_ylabel(f'd{nm}/dt')
        ax.set_title(f'{nm}  R²={r2:.3f}', fontsize=9)
        ax.legend(fontsize=7); ax.grid(alpha=0.3)
    axes[-1].set_xlabel('Week')
    plt.tight_layout(); plt.show()


def plot_heatmap(Xi, feat_names, state_names, title="Coefficient Matrix Ξ"):
    fig, ax = plt.subplots(figsize=(max(6,len(state_names)*1.5),
                                    max(4,len(feat_names)*0.32)))
    vm = max(np.abs(Xi).max(), 1e-6)
    im = ax.imshow(Xi, aspect='auto', cmap='RdBu_r', vmin=-vm, vmax=vm)
    ax.set_xticks(range(len(state_names)))
    ax.set_xticklabels([f'd{s}/dt' for s in state_names], rotation=45, ha='right')
    ax.set_yticks(range(len(feat_names)))
    ax.set_yticklabels(feat_names, fontsize=8)
    plt.colorbar(im, ax=ax); ax.set_title(title)
    plt.tight_layout(); plt.show()


def plot_aic_grid(results, k1, k2, metric='aic'):
    v1 = sorted(set(r['params'][k1] for r in results))
    v2 = sorted(set(r['params'][k2] for r in results))
    Z  = np.full((len(v1),len(v2)), np.nan)
    for r in results:
        i=v1.index(r['params'][k1]); j=v2.index(r['params'][k2])
        Z[i,j] = r[metric]
    fig, ax = plt.subplots(figsize=(6,4))
    im = ax.imshow(Z, aspect='auto', cmap='viridis', origin='lower')
    ax.set_xticks(range(len(v2))); ax.set_xticklabels(v2)
    ax.set_yticks(range(len(v1))); ax.set_yticklabels(v1)
    ax.set_xlabel(k2); ax.set_ylabel(k1)
    ax.set_title(f'{metric.upper()} surface')
    plt.colorbar(im,ax=ax,label=metric.upper())
    plt.tight_layout(); plt.show()

print("Diagnostics defined.")


## 8. V1 — Delayed SIR
*E absorbed into S̃; R̃(t) = I(t−τ)*

In [ ]:
print("=== V1: Delayed SIR ===")
best_v1, all_v1 = grid_search(
    lambda tau: build_V1(I_smooth, tau=tau),
    {'tau': TAU_GRID})

t1,X1,sn1 = best_v1['t'], best_v1['X'], best_v1['state_names']
Xi1,fn1,Th1,dX1,aic1,bic1 = run_sindy(t1, X1, sn1)

plot_compartments(t1, X1, sn1, "V1: Delayed SIR States")
plot_sindy_fit(t1, X1, Th1, Xi1, dX1, sn1, "V1: SINDy Fit")
plot_heatmap(Xi1, fn1, sn1, "V1: Coefficient Matrix")


## 9. V2 — Delayed SEIR
*E(t)≈I(t−τ₁); R from numerical integral*

In [ ]:
print("=== V2: Delayed SEIR ===")
best_v2, all_v2 = grid_search(
    lambda tau1, tau2: build_V2(I_smooth, tau1=tau1, tau2=tau2),
    {'tau1': TAU1_GRID, 'tau2': TAU2_GRID})

plot_aic_grid(all_v2, 'tau1', 'tau2')

t2,X2,sn2 = best_v2['t'], best_v2['X'], best_v2['state_names']
Xi2,fn2,Th2,dX2,aic2,bic2 = run_sindy(t2, X2, sn2)

plot_compartments(t2, X2, sn2, "V2: Delayed SEIR States")
plot_sindy_fit(t2, X2, Th2, Xi2, dX2, sn2, "V2: SINDy Fit")
plot_heatmap(Xi2, fn2, sn2, "V2: Coefficient Matrix")


## 10. V3 — Semi-analytic SEIR (Puspita et al. 2023)
*Multi-logistic K(t) fit + Eqs (11)–(14)*

In [ ]:
print("=== V3: Multi-logistic K fit ===")
p_opt, K_func, dK_func, rmse_K = fit_multi_logistic(t_raw, K)
print(f"Multi-logistic RMSE = {rmse_K:.4f}")

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t_raw, K, 'bo', ms=3, alpha=0.5, label='K data')
ax.plot(t_raw, K_func(t_raw), 'r-', lw=2, label=f'Fit (RMSE={rmse_K:.2f})')
ax.set_xlabel('Week'); ax.set_ylabel('K(t)')
ax.set_title('V3: Multi-logistic K(t) fit'); ax.legend()
plt.tight_layout(); plt.show()

t3,X3,sn3 = build_V3(I_smooth, K_func=K_func, dK_func=dK_func)
Xi3,fn3,Th3,dX3,aic3,bic3 = run_sindy(t3, X3, sn3)

plot_compartments(t3, X3, sn3, "V3: Semi-analytic SEIR States")
plot_sindy_fit(t3, X3, Th3, Xi3, dX3, sn3, "V3: SINDy Fit")
plot_heatmap(Xi3, fn3, sn3, "V3: Coefficient Matrix")


## 11. V4 — Semi-analytic SEIR + Cumulative C
*V3 + C(t)=∫I dt as auxiliary (dC/dt=I)*

In [ ]:
print("=== V4: Semi-analytic + C ===")
t4,X4,sn4 = build_V4(I_smooth, K_func=K_func, dK_func=dK_func)
Xi4,fn4,Th4,dX4,aic4,bic4 = run_sindy(t4, X4, sn4)

plot_compartments(t4, X4, sn4, "V4: Semi-analytic SEIR + C(t)")
plot_sindy_fit(t4, X4, Th4, Xi4, dX4, sn4, "V4: SINDy Fit")
plot_heatmap(Xi4, fn4, sn4, "V4: Coefficient Matrix")


## 12. V5 — Cumulative-only (no semi-analytic)
*States: [I, C, dC] — purely data-driven*

In [ ]:
print("=== V5: Cumulative-only ===")
t5,X5,sn5 = build_V5(I_smooth)
Xi5,fn5,Th5,dX5,aic5,bic5 = run_sindy(t5, X5, sn5)

plot_compartments(t5, X5, sn5, "V5: Cumulative-only States")
plot_sindy_fit(t5, X5, Th5, Xi5, dX5, sn5, "V5: SINDy Fit")
plot_heatmap(Xi5, fn5, sn5, "V5: Coefficient Matrix")


## 13. V6 — Delay Embedding (Takens)
*Reconstruct attractor from I(t) alone*

In [ ]:
print("=== V6: Delay Embedding ===")
best_v6, all_v6 = grid_search(
    lambda tau, embedding_dim: build_V6(I_smooth, tau=tau, embedding_dim=embedding_dim),
    {'tau': EMB_TAU, 'embedding_dim': EMB_DIM})

plot_aic_grid(all_v6, 'tau', 'embedding_dim')

t6,X6,sn6 = best_v6['t'], best_v6['X'], best_v6['state_names']
Xi6,fn6,Th6,dX6,aic6,bic6 = run_sindy(t6, X6, sn6)

plot_compartments(t6, X6, sn6, "V6: Delay Embedding States")
plot_sindy_fit(t6, X6, Th6, Xi6, dX6, sn6, "V6: SINDy Fit")
plot_heatmap(Xi6, fn6, sn6, "V6: Coefficient Matrix")


## 14. Variant Comparison

In [ ]:
summary = [
    ('V1 Delayed SIR',       aic1, bic1, best_v1['params']),
    ('V2 Delayed SEIR',      aic2, bic2, best_v2['params']),
    ('V3 Semi-analytic',     aic3, bic3, {}),
    ('V4 SA + C',            aic4, bic4, {}),
    ('V5 Cumulative-only',   aic5, bic5, {}),
    ('V6 Delay Embedding',   aic6, bic6, best_v6['params']),
]
summary.sort(key=lambda x: x[1])

print(f"{'Rank':<5} {'Variant':<22} {'AIC':>10} {'BIC':>10}  Best params")
print('-'*70)
for rank,(name,aic,bic,params) in enumerate(summary, 1):
    ps = ', '.join(f'{k}={v}' for k,v in params.items()) if params else '—'
    print(f"{rank:<5} {name:<22} {aic:>10.1f} {bic:>10.1f}  {ps}")

# Bar chart
names_ = [s[0] for s in summary]
aics_  = [s[1] for s in summary]
bics_  = [s[2] for s in summary]
x = np.arange(len(names_)); w = 0.35
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x-w/2, aics_, w, label='AIC', color='steelblue',   alpha=0.85)
ax.bar(x+w/2, bics_, w, label='BIC', color='darkorange', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(names_, rotation=20, ha='right')
ax.legend(); ax.set_ylabel('Score (lower = better)')
ax.set_title('Model Comparison: AIC / BIC across Variants')
plt.tight_layout(); plt.show()
